# Mapping

In [ ]:
# do contig fixing and mapping in two diff scripts

In [ ]:
#!/bin/bash
#SBATCH -c 8  # Number of Cores per Task
#SBATCH --mem=32G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 4:00:00  # Job time limit
#SBATCH --array=1-31%10
#SBATCH --mail-type=ALL,TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/binning/slurm-contigs-%A-%a.out

module load conda/latest
conda activate anvio-8

# array job - i think we need to do the array jobs based on the groups
SAMPLE_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_sample_groups.txt"
# using slurm array task sets the array jobs to this variable (can be named anything)
    # here i think it needs to be group so that it can rebuild the contigs together or else it will just repeat it 
INPUT_GROUP=$(tail -n +2 "$SAMPLE_FILE" | cut -f 3 |sort| uniq | sed -n "${SLURM_ARRAY_TASK_ID}p")

# GROUPTEST='122022_MCAV_Diseased_Margin'
# get spp from group 
SPP=$(grep "$INPUT_GROUP" "$SAMPLE_FILE" | awk '{print $2}' | uniq)


READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered"
CONTIGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/${INPUT_GROUP}/megahit_host_removed"
CONTIGFILE="${INPUT_GROUP}.contigs.fa"
WORKPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}"
mkdir -p "$WORKPATH"

XTRAFILES="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}/xtra"
mkdir -p "$XTRAFILES"

#fixes deflines (filters contigs and reformats so naming is cleaner)
#filtering seq length 1000bp...need to play around with filtering based on bp length
#deflines = sequence definition line. comes directly before its associated sequence in a fasta file
anvi-script-reformat-fasta "$CONTIGPATH/$CONTIGFILE" \
                            -o "$WORKPATH/${INPUT_GROUP}.contigs-fixed.fsa" \
                            -l 1000 \
                            --simplify-names \
                            --report-file "$WORKPATH/${INPUT_GROUP}contig-rename-report-txt"
FIXEDCON="${INPUT_GROUP}.contigs-fixed.fsa"

cd $WORKPATH
#this builds an index of your contigs, which only needs to happen once
bowtie2-build --threads "$SLURM_CPUS_PER_TASK" "$FIXEDCON" "${INPUT_GROUP}_contigs"
# will not accept path before contigs file - must be in the correct dir 

conda deactivate anvio-8
echo "Contigs fixed for ${INPUT_GROUP}"

# job id 
# job file: brooke /seqs042024 /mapping-contigs

In [ ]:
### testing with mcav first: 
sbatch --array=2-4,17-19,28 /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/mapping-contigs

In [ ]:
# diff script for mapping

In [ ]:
# test on mcav

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=62G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --array=1-67%10
#SBATCH --mail-type=ALL,TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/binning/slurm-mapping-%A-%a.out  # %j = job ID

module load conda/latest
conda activate anvio-8

# array job - i think we need to do the array jobs based on the groups
# sample ID list
SAMPLE_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_sample_groups.txt"
# input now needs to be sample
SAMPLEID=$(tail -n +2 "$SAMPLE_FILE" | grep "MCAV"| cut -f 1 | sed -n "${SLURM_ARRAY_TASK_ID}p")

# get group and species from sampleid
SPP=$(grep $SAMPLEID $SAMPLE_FILE | awk '{print $2}' | uniq)
INPUT_GROUP=$(grep $SAMPLEID $SAMPLE_FILE | awk '{print $3}' | uniq)

READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered"
CONTIGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/${INPUT_GROUP}/megahit_host_removed"
CONTIGFILE="${INPUT_GROUP}.contigs.fa"
WORKPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}"
XTRAFILES="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}/xtra"

# contig fixing script in previous bash script 
cd $WORKPATH || exit 1

R1="${READSPATH}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz"
R2="${READSPATH}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz"

#align reads to your contigs and collects that in a .sam file
bowtie2 --threads 14 -x "${INPUT_GROUP}_contigs" -1 "$R1" -2 "$R2" -S "$XTRAFILES/${SAMPLEID}.sam"
#make sure to point it to the index not the FIXEDCON file (-x parameter)

#converts your sam file to a bam file, but its neither sorted nor indexed, so we use an Anvi'O script to do so:
samtools view -F 4 -b -S -@ 8 "$XTRAFILES/${SAMPLEID}.sam" -o "$WORKPATH/${SAMPLEID}-RAW.bam"

#index and sort your bam file
anvi-init-bam -T 16 "$WORKPATH/${SAMPLEID}-RAW.bam" -o "$WORKPATH/${SAMPLEID}.bam"

rm "$XTRAFILES/${SAMPLEID}.sam"
rm "$WORKPATH/${SAMPLEID}-RAW.bam"

conda deactivate anvio-8
echo "Mapping success for ${SAMPLEID}!"

# job id
# job file: brooke /seqs042024 /mapping-reads-mcav.sh

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=64G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --array=1-222%10
#SBATCH --mail-type=ALL,TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/binning/slurm-mapping-%A-%a.out  # %j = job ID

module load conda/latest
conda activate anvio-8

# array job - i think we need to do the array jobs based on the groups
# sample ID list
SAMPLE_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_sample_groups.txt"
# input now needs to be sample
SAMPLEID=$(tail -n +2 "$SAMPLE_FILE" | cut -f 1 | sed -n "${SLURM_ARRAY_TASK_ID}p")

# get group and species from sampleid
SPP=$(grep $SAMPLEID $SAMPLE_FILE | awk '{print $2}' | uniq)
INPUT_GROUP=$(grep $SAMPLEID $SAMPLE_FILE | awk '{print $3}' | uniq)

READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered"
CONTIGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/${INPUT_GROUP}/megahit_host_removed"
CONTIGFILE="${INPUT_GROUP}.contigs.fa"
WORKPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}"
XTRAFILES="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/mapping/${INPUT_GROUP}/xtra"

# contig fixing script in previous bash script 
cd $WORKPATH || exit 1

R1="${READSPATH}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz"
R2="${READSPATH}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz"

#align reads to your contigs and collects that in a .sam file
bowtie2 --threads 14 -x "${INPUT_GROUP}_contigs" -1 "$R1" -2 "$R2" -S "$XTRAFILES/${SAMPLEID}.sam"
#make sure to point it to the index not the FIXEDCON file (-x parameter)

#converts your sam file to a bam file, but its neither sorted nor indexed, so we use an Anvi'O script to do so:
samtools view -F 4 -b -S -@ 8 "$XTRAFILES/${SAMPLEID}.sam" -o "$WORKPATH/${SAMPLEID}-RAW.bam"

#index and sort your bam file
anvi-init-bam -T 16 "$WORKPATH/${SAMPLEID}-RAW.bam" -o "$WORKPATH/${SAMPLEID}.bam"

rm "$XTRAFILES/${SAMPLEID}.sam"
rm "$WORKPATH/${SAMPLEID}-RAW.bam"

conda deactivate anvio-8
echo "Mapping success for ${SAMPLEID}!"

# job id
# job file: brooke /seqs042024 /mapping-reads